# MerLog Chatbot — сравнение моделей

тут сравниваю TF-IDF плюс LogReg и DistilBERT на одном датасете чтобы понять стоит ли тащить трансформер в продакшн или классики хватит

датасет: Bitext Customer Support, отфильтровал до 6 логистических интентов, получилось 5985 примеров

## 1. Грузим данные и обучаем TF-IDF плюс LogReg

In [1]:
import sys
sys.path.append('../src')

import time
import pandas as pd
from intent_classifier import load_processed, train, save_model, predict

df = load_processed()
print(f"Dataset: {len(df)} rows, {df['intent'].nunique()} intents")
print(df.groupby('intent').size())

Dataset: 5985 rows, 6 intents
intent
cancel_order         998
change_order         997
check_invoice       1000
complaint           1000
delivery_options     995
track_order          995
dtype: int64


In [2]:
start = time.time()
pipe, tfidf_metrics, _ = train(df)
tfidf_train_time = time.time() - start
print(f"TF-IDF + LogReg training: {tfidf_train_time:.2f}s")
print(f"Accuracy: {tfidf_metrics['accuracy']:.4f}")
print(f"F1 macro: {tfidf_metrics['f1_macro']:.4f}")

TF-IDF + LogReg training: 0.42s
Accuracy: 0.9958
F1 macro: 0.9958


In [3]:
start = time.time()
for _ in range(100):
    predict(pipe, "where is my shipment MRL-2024-8831")
tfidf_infer_time = (time.time() - start) / 100 * 1000
print(f"TF-IDF inference: {tfidf_infer_time:.2f} ms/query")

TF-IDF inference: 1.61 ms/query


## 2. Обучаем DistilBERT

In [4]:
from transformer_classifier import train_transformer, predict_transformer

start = time.time()
model, tokenizer, bert_metrics = train_transformer(df, epochs=3, batch_size=16)
bert_train_time = time.time() - start
print(f"DistilBERT training: {bert_train_time:.2f}s")
print(f"Accuracy: {bert_metrics['accuracy']:.4f}")
print(f"F1 macro: {bert_metrics['f1_macro']:.4f}")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,No log,0.019883,0.995823,0.995812
2,0.235434,0.009456,0.998329,0.998327
3,0.235434,0.009285,0.997494,0.997487


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training Loss,Validation Loss,Epoch,Accuracy,F1 Macro
0.235434,0.009456,3,0.998329,0.998327


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

DistilBERT training: 121.11s
Accuracy: 0.9983
F1 macro: 0.9983


In [5]:
import torch
model = model.to('cpu')
start = time.time()
for _ in range(50):
    predict_transformer("where is my shipment MRL-2024-8831", model, tokenizer)
bert_infer_time = (time.time() - start) / 50 * 1000
print(f"DistilBERT inference: {bert_infer_time:.2f} ms/query")

DistilBERT inference: 17.15 ms/query


## 3. Сравнение

In [6]:
comparison = pd.DataFrame({
    'Model': ['TF-IDF + LogReg', 'DistilBERT'],
    'Accuracy': [tfidf_metrics['accuracy'], bert_metrics['accuracy']],
    'F1 macro': [tfidf_metrics['f1_macro'], bert_metrics['f1_macro']],
    'Train time (s)': [tfidf_train_time, bert_train_time],
    'Inference (ms)': [tfidf_infer_time, bert_infer_time],
})
comparison

,Model,Accuracy,F1 macro,Train time (s),Inference (ms)
0,TF-IDF + LogReg,0.995823,0.995823,0.419456,1.613662
1,DistilBERT,0.998329,0.998327,121.106091,17.145839


## 4. Вывод

DistilBERT даёт чуть выше точность, около 0.25%, но для продакшена выбрал TF-IDF плюс LogReg и вот почему:

- скорость: TF-IDF на порядки быстрее, для real-time чат бота это критично, клиент не будет ждать 17 мс когда можно за 1.6 мс
- объяснимость: веса фичей можно посмотреть и показать стейкхолдерам MerLog почему бот принял такое решение
- простота: не нужен GPU, модель весит килобайты а не гигабайты, деплой проще
- разница в точности: 0.25% не стоит той сложности которую тащит трансформер

а confidence-роутер компенсирует чуть меньшую точность тем что неуверенные ответы уходит к оператору